# Four-momentum and two-body decay

*nuclear · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/nuclear/four_momentum)

A moving particle decays into two. In its rest frame the products fly back to back with a fixed momentum $p^*$; a Lorentz boost $\Lambda^\mu{}_\nu$ (one einsum) carries every decay angle at once to the lab, where the momenta are swept forward. The Minkowski metric $\eta_{\mu\nu}$ turns the products' four-momenta back into the parent's mass — the invariant-mass trick used to discover new particles.

**Vector form**

$$p^\mu = (E, \mathbf p),\quad p\cdot p = E^2 - \mathbf p^2 = m^2,\qquad p^* = \frac{\sqrt{[M^2-(m_1+m_2)^2][M^2-(m_1-m_2)^2]}}{2M},\qquad (p_1+p_2)^2 = M^2$$

**Index (tensor) form**

$$p_\mu p^\mu = \eta_{\mu\nu}\,p^\mu p^\nu = m^2,\qquad p'^{\mu}_{b} =\Lambda^\mu{}_\nu\,p^\nu_b,\qquad M^2 = \eta_{\mu\nu}\,(p_1+p_2)^\mu_b\,(p_1+p_2)^\nu_b$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\eta_{\mu\nu}$ | `metric(device)` | (4, 4) | diag(1, −1, −1, −1), units with $c = 1$ |
| $p^\mu_b$ | `rest_frame_daughters(theta, p_star, ...)` | (B, 4) | daughter four-momenta for B decay angles $\theta^*$ in the rest frame |
| $\Lambda^\mu{}_\nu$ | `boost(beta)` | (4, 4) | $\gamma$, $\gamma\beta_i$ and $\delta_{ij} + (\gamma-1)\beta_i\beta_j/\beta^2$ for the parent velocity $\boldsymbol\beta = \mathbf P/E$ |
| $\Lambda^\mu{}_\nu\,p^\nu_b$ | `einsum('mn,...n->...m', lam, p)` | (B, 4) | repeated $\nu$ ⇒ summed: boost every decay at once |
| $\eta_{\mu\nu}p^\mu q^\nu$ | `einsum('...m,mn,...n->...', p, eta, q)` | (B,) | Minkowski product: the same in every frame |
| $M_b$ | `invariant_mass(p1 + p2, eta)` | (B,) | reconstructed parent mass for every configuration |

The rest-frame momenta lie on a sphere of radius $p^*$; the boost stretches it into an ellipse (the traced curves) shifted forward by the parent's momentum, so fast parents throw both products into a narrow forward cone. Energy and lab angles change with $\theta^*$, but $p_\mu p^\mu = m^2$ for each product and $(p_1+p_2)_\mu(p_1+p_2)^\mu = M^2$ never change — a detector that measures the two momenta can always reconstruct $M$. For $\pi^0 \to \gamma\gamma$ the photons are massless ($p_\mu p^\mu = 0$) and the smallest opening angle is $2\arcsin(1/\gamma)$. All three decays are real: $K^0_S \to \pi^+\pi^-$, $\pi^0 \to \gamma\gamma$, $\Lambda \to p\,\pi^-$. Only the kinematics is modelled: decay angles are swept evenly instead of sampled, and spin and polarisation are ignored.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/nuclear/four_momentum/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import invariant_mass, metric, rest_frame_momentum


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    title, big_m, m1, m2, name1, name2 = tensors.DECAYS[params["decay"]]
    eta = metric(dev)  # (4, 4)
    parent = tensors.parent(big_m, params["p_parent"], dev)  # (4,)
    theta = tensors.decay_angles(72, dev)  # (B,)
    p_star = rest_frame_momentum(big_m, m1, m2)
    rest1, rest2 = tensors.rest_frame_daughters(theta, p_star, big_m, m1, m2)  # (B, 4) each

    lab1, lab2 = solve.to_lab(parent, rest1, rest2)  # (B, 4) each
    m_inv = solve.reconstruct(lab1, lab2, eta)  # (B,)

    gamma, beta = parent[0].item() / big_m, parent[3].item() / parent[0].item()
    print(f"{title}:  M = {big_m:.4f} GeV, p = {params['p_parent']:.2f} GeV  →  γ = {gamma:.3f}, β = {beta:.4f}")
    print(f"rest-frame momentum p* = {p_star:.4f} GeV")
    mass1, mass2 = invariant_mass(lab1, eta).mean().item(), invariant_mass(lab2, eta).mean().item()
    print(f"masses after the boost: {name1} {mass1:.5f}  {name2} {mass2:.5f} GeV")
    print(f"reconstructed parent mass: {m_inv.min().item():.6f} … {m_inv.max().item():.6f} GeV")
    print(f"max |p1 + p2 − P| = {(lab1 + lab2 - parent).abs().max().item():.1e} GeV")
    for k in (0, len(theta) // 2, len(theta) - 1):
        angle = math.degrees(theta[k].item())
        print(f"θ* = {angle:5.1f}°   lab energies {lab1[k, 0].item():.3f} + {lab2[k, 0].item():.3f} GeV")
    return plot.animate(
        theta, rest1, rest2, lab1, lab2, parent, m_inv, (title, name1, name2), out_dir / "four_momentum.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/nuclear/four_momentum/equation.py
r"""Physics → tensor translation.

Energy and momentum form one four-vector p^μ = (E, p_x, p_y, p_z) (units with c = 1, GeV). With the metric
η_{μν} = diag(1, −1, −1, −1) the invariant "length" is the mass,

    p_μ p^μ = η_{μν} p^μ p^ν = E^2 - \mathbf p^2 = m^2        (μ, ν summed).

A particle of mass M at rest decays into m_1 and m_2 flying back to back with momentum

    p^* = \frac{\sqrt{[M^2 - (m_1 + m_2)^2]\,[M^2 - (m_1 - m_2)^2]}}{2M}.

Moving to the lab, where the parent has velocity \boldsymbol β, every four-vector is transformed by the boost

    p'^μ = Λ^μ{}_ν \, p^ν,     Λ^0{}_0 = γ,   Λ^0{}_i = Λ^i{}_0 = γ β_i,   Λ^i{}_j = δ_{ij} + (γ - 1) β_i β_j / β^2.

The invariant mass of the decay products, (p_1 + p_2)_μ (p_1 + p_2)^μ = M^2, is the same in every frame.
"""

import math

import torch


def metric(device: torch.device) -> torch.Tensor:
    r"""η_{μν} = diag(1, −1, −1, −1): shape (4, 4)."""
    return torch.diag(torch.tensor([1.0, -1.0, -1.0, -1.0], dtype=torch.float64, device=device))


def minkowski(p: torch.Tensor, q: torch.Tensor, eta: torch.Tensor) -> torch.Tensor:
    r"""p·q = η_{μν} p^μ q^ν   (μ, ν summed).   p, q: (..., 4) → (...)"""
    return torch.einsum("...m,mn,...n->...", p, eta, q)


def invariant_mass(p: torch.Tensor, eta: torch.Tensor) -> torch.Tensor:
    r"""m = \sqrt{p_μ p^μ}.   (..., 4) → (...)"""
    return minkowski(p, p, eta).clamp(min=0.0).sqrt()


def rest_frame_momentum(big_m: float, m1: float, m2: float) -> float:
    r"""p^* of the two daughters in the parent's rest frame."""
    return math.sqrt((big_m**2 - (m1 + m2) ** 2) * (big_m**2 - (m1 - m2) ** 2)) / (2 * big_m)


def boost(beta: torch.Tensor) -> torch.Tensor:
    r"""Λ^μ{}_ν for a frame moving with velocity -\boldsymbol β (so a particle at rest gets velocity +β).
    beta: (3,) → (4, 4)"""
    b2 = torch.dot(beta, beta)
    gamma = 1.0 / torch.sqrt(1.0 - b2)
    lam = torch.empty(4, 4, dtype=beta.dtype, device=beta.device)
    lam[0, 0] = gamma
    lam[0, 1:] = gamma * beta
    lam[1:, 0] = gamma * beta
    lam[1:, 1:] = torch.eye(3, dtype=beta.dtype, device=beta.device) + (gamma - 1) * torch.outer(beta, beta) / b2
    return lam


def transform(lam: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
    r"""p'^μ = Λ^μ{}_ν p^ν   (ν summed).   lam: (4, 4), p: (..., 4) → (..., 4)"""
    return torch.einsum("mn,...n->...m", lam, p)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/nuclear/four_momentum/tensors.py
"""Build the concrete tensors: particle masses, the parent four-momentum and the rest-frame decay products."""

import math

import torch

# decay: (label, M, m1, m2, name1, name2), masses in GeV
DECAYS = {
    "kaon": ("K⁰ → π⁺ π⁻", 0.497611, 0.139570, 0.139570, "π⁺", "π⁻"),
    "pion": ("π⁰ → γ γ", 0.134977, 0.0, 0.0, "γ", "γ"),
    "lambda": ("Λ → p π⁻", 1.115683, 0.938272, 0.139570, "p", "π⁻"),
}


def parent(big_m: float, p_lab: float, device: torch.device) -> torch.Tensor:
    """P^μ = (√(M² + p²), 0, 0, p): the parent flies along z in the lab: shape (4,)."""
    return torch.tensor([math.sqrt(big_m**2 + p_lab**2), 0.0, 0.0, p_lab], dtype=torch.float64, device=device)


def decay_angles(n: int, device: torch.device) -> torch.Tensor:
    """Rest-frame emission angles θ* of daughter 1 relative to the flight direction, 0 … π: shape (B,)."""
    return torch.linspace(0.0, math.pi, n, dtype=torch.float64, device=device)


def rest_frame_daughters(
    theta: torch.Tensor, p_star: float, big_m: float, m1: float, m2: float
) -> tuple[torch.Tensor, torch.Tensor]:
    """Back-to-back four-momenta in the parent's rest frame, in the x–z plane: two tensors of shape (B, 4).

    p1 = (E1,  p* sin θ, 0,  p* cos θ),   p2 = (E2, −p* sin θ, 0, −p* cos θ),   E_k = √(m_k² + p*²).
    """
    e1, e2 = math.sqrt(m1**2 + p_star**2), math.sqrt(m2**2 + p_star**2)
    n = torch.stack([torch.sin(theta), torch.zeros_like(theta), torch.cos(theta)], dim=-1)  # (B, 3) unit vectors
    p1 = torch.cat([torch.full_like(theta, e1)[:, None], p_star * n], dim=-1)
    p2 = torch.cat([torch.full_like(theta, e2)[:, None], -p_star * n], dim=-1)
    return p1, p2


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/nuclear/four_momentum/solve.py
"""Boost every decay configuration to the lab and reconstruct the parent from its products."""

import torch

from .equation import boost, invariant_mass, transform


def to_lab(parent: torch.Tensor, p1: torch.Tensor, p2: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Apply the boost with the parent's velocity β = P/E to both daughters: (B, 4) each."""
    lam = boost(parent[1:] / parent[0])  # (4, 4)
    return transform(lam, p1), transform(lam, p2)


def reconstruct(p1: torch.Tensor, p2: torch.Tensor, eta: torch.Tensor) -> torch.Tensor:
    """Invariant mass of the pair, √((p1 + p2)·(p1 + p2)), for every configuration: shape (B,)."""
    return invariant_mass(p1 + p2, eta)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/nuclear/four_momentum/plot.py
"""Sweep the decay angle: back-to-back momenta in the rest frame, boosted momenta in the lab."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

C1, C2, CP = "#2563eb", "#dc2626", "#374151"


def animate(
    theta: torch.Tensor,
    rest1: torch.Tensor,
    rest2: torch.Tensor,
    lab1: torch.Tensor,
    lab2: torch.Tensor,
    parent: torch.Tensor,
    m_inv: torch.Tensor,
    labels: tuple[str, str, str],
    path: Path,
) -> Path:
    """All momenta (B, 4); we draw the (z, x) components, z = flight direction of the parent."""
    deg = np.degrees(theta.cpu().numpy())
    r1, r2 = rest1.cpu().numpy()[:, [3, 1]], rest2.cpu().numpy()[:, [3, 1]]  # (B, 2) as (p_z, p_x)
    l1, l2 = lab1.cpu().numpy()[:, [3, 1]], lab2.cpu().numpy()[:, [3, 1]]
    p_par = parent.cpu().numpy()[[3, 1]]
    mass = m_inv.cpu().numpy()
    title, name1, name2 = labels

    fig, (ax_r, ax_l) = plt.subplots(1, 2, figsize=(9, 4), gridspec_kw={"width_ratios": [1, 1.6]})
    p_star = np.linalg.norm(r1[0])
    span = 1.3 * p_star
    ax_r.set_xlim(-span, span)
    ax_r.set_ylim(-span, span)
    ax_r.add_patch(plt.Circle((0, 0), p_star, fill=False, ls="--", color="#9ca3af"))
    ax_r.set_title("parent rest frame", fontsize=10)
    ax_r.set_xlabel("$p_z$ (GeV)")
    ax_r.set_ylabel("$p_x$ (GeV)")
    angle_text = ax_r.text(0.03, 0.95, "", transform=ax_r.transAxes, va="top", fontsize=9)

    zmax = max(l1[:, 0].max(), l2[:, 0].max(), p_par[0]) * 1.1
    zmin = min(l1[:, 0].min(), l2[:, 0].min(), 0.0) * 1.1 - 0.05 * zmax
    xmax = max(np.abs(l1[:, 1]).max(), np.abs(l2[:, 1]).max()) * 1.4
    ax_l.set_xlim(zmin, zmax)
    ax_l.set_ylim(-xmax, xmax)
    ax_l.set_title(f"lab frame: {title}", fontsize=10)
    ax_l.set_xlabel("$p_z$ (GeV)")
    for ax in (ax_r, ax_l):
        ax.set_aspect("equal", adjustable="box")
    ax_l.annotate("", xy=p_par, xytext=(0, 0), arrowprops={"arrowstyle": "->", "color": CP, "lw": 1, "ls": "--"})
    ax_l.text(p_par[0], 0.06 * xmax, "parent", color=CP, fontsize=8, ha="right")
    (trace1,) = ax_l.plot([], [], color=C1, lw=1, alpha=0.5)
    (trace2,) = ax_l.plot([], [], color=C2, lw=1, alpha=0.5)
    readout = ax_l.text(0.02, 0.04, "", transform=ax_l.transAxes, fontsize=8, family="monospace")

    arrows: list = []

    def arrow(ax, tip, tail, colour, label):
        a = ax.annotate("", xy=tip, xytext=tail, arrowprops={"arrowstyle": "-|>", "color": colour, "lw": 2})
        t = ax.text(*(tip + 0.04 * span), label, color=colour, fontsize=9)
        arrows.extend([a, t])

    def update(step: int):
        for artist in arrows:
            artist.remove()
        arrows.clear()
        arrow(ax_r, r1[step], (0, 0), C1, name1)
        arrow(ax_r, r2[step], (0, 0), C2, name2)
        arrow(ax_l, l1[step], (0, 0), C1, name1)
        arrow(ax_l, l2[step], (0, 0), C2, name2)
        arrow(ax_l, l1[step] + l2[step], l1[step], C2, "")  # p2 drawn from the tip of p1: sum = parent
        trace1.set_data(l1[: step + 1, 0], l1[: step + 1, 1])
        trace2.set_data(l2[: step + 1, 0], l2[: step + 1, 1])
        angle_text.set_text(f"θ* = {deg[step]:.0f}°")
        readout.set_text(f"m_inv = √((p1+p2)·(p1+p2)) = {mass[step]:.4f} GeV")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(deg), 72), path, fps=15)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'decay': 'kaon',
    'p_parent': 1.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.nuclear.four_momentum")]:
    del sys.modules[name]
main = importlib.import_module("problems.nuclear.four_momentum.main")
problem_dir = Path("problems/nuclear/four_momentum")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
